In [ ]:
# ============================================================================
#
#    RARE DISEASE CLINICAL TRIAL PROTOCOL GENERATOR
#    LLM-Based Protocol Generation
#
# ============================================================================
#
# PROJECT: AI-Driven Clinical Trial Protocol Development for Rare Diseases
# Protocol Generation & Testing
# LLM MODEL: Mistral 7B (Open-Source)
#
# OBJECTIVE:
#   Generate standardized, high-quality clinical trial protocols for
#   rare diseases using large language models with cross-system
#   biomarker integration.
#
# OUTPUT:
#   15 protocols (3 per disease × 5 diseases)
#   Format: CSV with structured protocol data
#   Location: Google Drive > Results folder
#
# PARAMETERS:
#   • Diseases: 5 (Fabry, Dravet, Autoimmune Encephalitis, DIPG, Autism+Microbiome)
#   • Protocols per disease: 3
#   • Total protocols: 15
#   • Estimated runtime: 40-50 minutes
#   • GPU: Required (enabled in Colab Pro)
#
# AUTHOR: Dr. Hameem Mahdi (Senior Principal Applied Scientist)
# DATE: February 2026
# VERSION: 1.0
#
# REQUIREMENTS:
#   ✓ Google Colab Pro (GPU access)
#   ✓ Google Drive mounted
#   ✓ transformers library
#   ✓ torch
#   ✓ pandas
#
# ===========================================================================

In [ ]:
# ============================================================================
# Mount Google Drive & Set Up Complete Project Structure
# ============================================================================

from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')
print("Google Drive mounted successfully!\n")

# Set up project path
project_path = "/content/drive/My Drive/Colab Notebooks/research_codebase/LLM-Generated Protocols for Rare Diseases/Phase 2 - LLM & Prompt Engineering"

# Change to project directory
os.chdir(project_path)
print(f"Working directory: {os.getcwd()}\n")

# Create folder structure
folders = [
    'Results'
]

print("Creating folder structure...")
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"   Created: {folder}/")

print("\n" + "="*70)
print("PROJECT SETUP COMPLETE")
print("="*70)
print(f"Working Directory: {os.getcwd()}")
print(f"\nFolders created:")
for folder in folders:
    print(f"  • {folder}/")
print(f"\nDefault save location: {os.getcwd()}/Results/")
print("="*70)

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!

Working directory: /content/drive/My Drive/Colab Notebooks/research_codebase/LLM-Generated Protocols for Rare Diseases/Phase 2 - LLM & Prompt Engineering

Creating folder structure...
   Created: Results/

PROJECT SETUP COMPLETE
Working Directory: /content/drive/My Drive/Colab Notebooks/research_codebase/LLM-Generated Protocols for Rare Diseases/Phase 2 - LLM & Prompt Engineering

Folders created:
  • Results/

Default save location: /content/drive/My Drive/Colab Notebooks/research_codebase/LLM-Generated Protocols for Rare Diseases/Phase 2 - LLM & Prompt Engineering/Results/


In [ ]:
# ============================================================================
# STEP 1: Install Dependencies
!pip install -q transformers torch pandas tqdm

In [ ]:
# ============================================================================
# STEP 2: Import Libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from tqdm import tqdm
import time

print("Libraries imported successfully")
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

Libraries imported successfully
GPU Available: True
GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:
# ============================================================================
# STEP 3: Load Mistral 7B Model
print("\nLoading Mistral 7B model (this takes ~2-3 minutes)...")
print("This happens only once. Subsequent cells will be faster.\n")

model_name = "mistralai/Mistral-7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # Use float16 for faster processing on GPU
    device_map="auto"
)

print("Model loaded successfully!")
print(f"   Model: {model_name}")
print(f"   Parameters: 7B")


Loading Mistral 7B model (this takes ~2-3 minutes)...
This happens only once. Subsequent cells will be faster.



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully!
   Model: mistralai/Mistral-7B-Instruct-v0.1
   Parameters: 7B


In [ ]:
# ============================================================================
# STEP 4: Define Rare Diseases
# These are your 5 diseases from the research paper

diseases = {
    "Fabry Disease": {
        "description": "X-linked lysosomal storage disorder caused by GLA gene mutation. Affects heart, kidney, and nervous system.",
        "organs": ["Cardiovascular", "Renal", "Neurological"],
        "prevalence": "1 in 40,000 to 1 in 117,000",
        "current_treatment": "Enzyme replacement therapy (imiglucerase, agalsidase beta), substrate reduction therapy"
    },
    "Dravet Syndrome": {
        "description": "Severe developmental and epileptic encephalopathy caused by SCN1A mutations. Presents with prolonged febrile seizures.",
        "organs": ["Neurological", "Neurodevelopmental"],
        "prevalence": "1 in 20,000 to 1 in 40,000",
        "current_treatment": "Valproic acid, levetiracetam, clobazam; recently approved: stiripentol, fenfluramine"
    },
    "Autoimmune Encephalitis": {
        "description": "Immune-mediated CNS inflammation often associated with antibodies against neuronal surface antigens (NMDA, AMPA, LGI1).",
        "organs": ["Neurological", "Immunological"],
        "prevalence": "1-3 per million per year",
        "current_treatment": "Immunotherapy: corticosteroids, IVIG, plasma exchange; monoclonal antibodies (rituximab, tocilizumab)"
    },
    "Pediatric DIPG": {
        "description": "Diffuse Intrinsic Pontine Glioma. Malignant brainstem tumor in children. H3K27M mutation common.",
        "organs": ["Neurological", "Oncological"],
        "prevalence": "1 in 100,000 pediatric cancer cases",
        "current_treatment": "Focal radiation therapy; chemotherapy regimens being investigated; targeted therapies (H3K27M inhibitors in development)"
    },
    "Autism with Microbiome Dysbiosis": {
        "description": "Autism spectrum disorder associated with altered gut microbiota composition. Emerging evidence for microbiome-gut-brain axis involvement.",
        "organs": ["Neurological", "Gastrointestinal", "Immunological"],
        "prevalence": "Autism: 1 in 36-38 children",
        "current_treatment": "Behavioral interventions (ABA); medications for comorbidities; emerging: probiotic/prebiotic trials"
    }
}

print("Diseases defined:")
for disease in diseases.keys():
    print(f"   • {disease}")

Diseases defined:
   • Fabry Disease
   • Dravet Syndrome
   • Autoimmune Encephalitis
   • Pediatric DIPG
   • Autism with Microbiome Dysbiosis


In [ ]:
# ============================================================================
# STEP 5: Define Standardized Prompt Template

def create_prompt(disease_name, disease_info, protocol_number):
    """
    Create a standardized prompt for protocol generation
    """
    prompt = f"""You are an expert clinical trial protocol designer specializing in rare diseases.

Generate a comprehensive clinical trial protocol for the following rare disease:

DISEASE: {disease_name}
Description: {disease_info['description']}
Affected Organ Systems: {', '.join(disease_info['organs'])}
Patient Population Prevalence: {disease_info['prevalence']}
Current Standard Treatment: {disease_info['current_treatment']}

PROTOCOL REQUIREMENTS:

1. DISEASE UNDERSTANDING SECTION:
   - Provide detailed pathophysiology of {disease_name}
   - Explain organ system involvement and cross-system interactions
   - Justify why this disease needs a new clinical trial
   - Discuss current treatment gaps

2. RESEARCH ENDPOINTS SECTION:
   - Define PRIMARY ENDPOINT (specific, measurable, clinically meaningful)
   - Define 2-3 SECONDARY ENDPOINTS
   - Include EXPLORATORY ENDPOINTS (biomarkers, cross-system measures)
   - For cross-system diseases: Include biomarkers from multiple affected organs
   - Justify each endpoint with explicit reasoning

3. STUDY DESIGN SECTION:
   - Specify: Phase (Phase 2, Phase 3, etc.)
   - Specify: Design type (randomized controlled trial, adaptive design, etc.)
   - Duration of study and treatment period
   - Rationale for design choice given rare disease constraints

4. SAMPLE SIZE & POPULATION SECTION:
   - Calculate estimated sample size (show calculation)
   - Justify sample size considering patient population prevalence
   - Define inclusion/exclusion criteria
   - Discuss recruitment strategy for rare disease population
   - Be realistic: Account for global patient population of {disease_info['prevalence']}

5. ADAPTIVE DESIGN & MODERN METHODS SECTION:
   - Include adaptive design elements (if appropriate for disease)
   - Incorporate data science advances: automated EHR analysis, AI-assisted diagnosis, wearable monitoring
   - For neurological diseases: Include neuroimaging or EEG analysis
   - For immunological diseases: Include molecular profiling, flow cytometry
   - For multi-system diseases: Integrate biomarkers across systems

6. CROSS-SYSTEM BIOMARKER INTEGRATION (if applicable):
   - If disease affects multiple organs: Define how biomarkers from different systems connect
   - Explain mechanistic links between organ systems
   - Justify why measuring BOTH systems is necessary
   - Propose integrated analysis strategy

7. SAFETY MONITORING SECTION:
   - Specify safety monitoring plan
   - Include stopping rules for futility/safety
   - Define serious adverse events specific to {disease_name}
   - Include organ-system specific safety measures

8. QUALITY & FEASIBILITY SECTION:
   - Ensure protocol is realistic given rare disease constraints
   - Do NOT invent non-existent drugs or biomarkers
   - Use ONLY real, validated biomarkers
   - Do NOT cite fake regulatory guidelines

OUTPUT FORMAT:
- Structured sections with clear headers
- Professional clinical trial protocol language
- Length: 1500-2500 words
- Include explicit justifications for major design decisions
- This is PROTOCOL #{protocol_number} for {disease_name}

Begin protocol:
"""
    return prompt

print("Prompt template created")

Prompt template created


In [ ]:
# ============================================================================
# STEP 6: Generate Protocols Function

def generate_protocol(disease_name, disease_info, protocol_number, max_length=2500):
    """
    Generate a single protocol using Mistral 7B
    """
    prompt = create_prompt(disease_name, disease_info, protocol_number)

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate output
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        temperature=0.7,  # Some creativity but not too random
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode output
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract just the protocol part (remove the prompt)
    protocol_text = generated_text[len(prompt):]

    return protocol_text.strip()

print("Generation function ready")

Generation function ready


In [ ]:
# ============================================================================
# STEP 7: Generate All 15 Protocols
# ============================================================================

from tqdm.auto import tqdm
import time
import torch

print("\n" + "="*70)
print("STARTING PROTOCOL GENERATION")
print("="*70)
print(f"Generating: 3 protocols × 5 diseases = 15 total protocols")
print(f"Estimated time: 30-40 minutes")
print("="*70 + "\n")

# Storage for results
all_protocols = []
total_protocols = len(diseases) * 3
protocol_count = 0
start_time = time.time()

# Progress bar
pbar = tqdm(total=total_protocols, desc="Overall Progress", unit=" protocol")

for disease_name, disease_info in diseases.items():

    for protocol_num in range(1, 4):
        protocol_count += 1

        pbar.set_description(f"{disease_name} ({protocol_num}/3)")

        try:
            gen_start = time.time()

            # Use existing generate_protocol function from Cell 6
            protocol_text = generate_protocol(disease_name, disease_info, protocol_num)

            gen_elapsed = time.time() - gen_start

            # Validate
            word_count = len(protocol_text.split())
            char_count = len(protocol_text)
            alpha_count = sum(c.isalpha() for c in protocol_text)
            alpha_ratio = alpha_count / char_count if char_count > 0 else 0

            # Check quality
            if alpha_ratio < 0.3 or word_count < 100:
                pbar.write(f"  [WARNING] Low quality ({word_count} words) - Regenerating...")
                protocol_text = generate_protocol(disease_name, disease_info, protocol_num)
                word_count = len(protocol_text.split())
                char_count = len(protocol_text)
                alpha_count = sum(c.isalpha() for c in protocol_text)
                alpha_ratio = alpha_count / char_count if char_count > 0 else 0

            # Store result
            protocol_entry = {
                "Disease": disease_name,
                "Protocol_Number": protocol_num,
                "LLM": "Mistral-7B",
                "Protocol_ID": f"{disease_name.replace(' ', '_')}_{protocol_num}",
                "Protocol_Text": protocol_text,
                "Word_Count": word_count,
                "Generation_Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "Alpha_Ratio": round(alpha_ratio, 3),
                "Validated": "PASS" if alpha_ratio > 0.3 and word_count > 100 else "FAIL"
            }

            all_protocols.append(protocol_entry)

            # Time estimates
            elapsed_total = time.time() - start_time
            avg_time = elapsed_total / protocol_count
            remaining = (total_protocols - protocol_count) * avg_time

            status = "PASS" if protocol_entry["Validated"] == "PASS" else "WARN"
            pbar.write(f"  [{protocol_count:2d}/15] {disease_name} Protocol {protocol_num} | {status} | {word_count:4d} words | {gen_elapsed:3.0f}s | Remaining: {remaining/60:4.1f}m")

            # Clear cache
            torch.cuda.empty_cache()

        except Exception as e:
            pbar.write(f"  [ERROR] {disease_name} Protocol {protocol_num}: {str(e)}")
            all_protocols.append({
                "Disease": disease_name,
                "Protocol_Number": protocol_num,
                "LLM": "Mistral-7B",
                "Protocol_ID": f"{disease_name.replace(' ', '_')}_{protocol_num}",
                "Protocol_Text": f"[ERROR: {str(e)}]",
                "Word_Count": 0,
                "Generation_Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "Alpha_Ratio": 0.0,
                "Validated": "ERROR"
            })

        pbar.update(1)

pbar.close()

elapsed_total = time.time() - start_time

print(f"\n{'='*70}")
print(f"GENERATION COMPLETE")
print(f"{'='*70}")
print(f"Total protocols generated: {len(all_protocols)}")
print(f"Total time: {elapsed_total/60:.1f} minutes")
print(f"Average time per protocol: {elapsed_total/len(all_protocols):.1f} seconds")

# Quality summary
df_temp = pd.DataFrame(all_protocols)
passed = len(df_temp[df_temp['Validated'] == 'PASS'])
failed = len(df_temp[df_temp['Validated'] != 'PASS'])

print(f"\nQuality Summary:")
print(f"  Passed: {passed}")
print(f"  Failed: {failed}")


STARTING PROTOCOL GENERATION
Generating: 3 protocols × 5 diseases = 15 total protocols
Estimated time: 30-40 minutes



Overall Progress:   0%|          | 0/15 [00:00<?, ? protocol/s]

  [ 1/15] Fabry Disease Protocol 1 | PASS | 1024 words |  37s | Remaining:  8.6m
  [ 2/15] Fabry Disease Protocol 2 | PASS | 1362 words |  37s | Remaining:  8.0m
  [ 3/15] Fabry Disease Protocol 3 | PASS | 1468 words |  35s | Remaining:  7.3m
  [ 4/15] Dravet Syndrome Protocol 1 | PASS | 1523 words |  37s | Remaining:  6.7m
  [ 5/15] Dravet Syndrome Protocol 2 | PASS | 1448 words |  36s | Remaining:  6.1m
  [ 6/15] Dravet Syndrome Protocol 3 | PASS | 1170 words |  29s | Remaining:  5.3m
  [ 7/15] Autoimmune Encephalitis Protocol 1 | PASS | 1528 words |  37s | Remaining:  4.7m
  [ 8/15] Autoimmune Encephalitis Protocol 2 | PASS | 1544 words |  37s | Remaining:  4.2m
  [ 9/15] Autoimmune Encephalitis Protocol 3 | PASS | 1493 words |  37s | Remaining:  3.6m
  [10/15] Pediatric DIPG Protocol 1 | PASS |  586 words |  13s | Remaining:  2.8m
  [11/15] Pediatric DIPG Protocol 2 | PASS | 1607 words |  37s | Remaining:  2.3m
  [12/15] Pediatric DIPG Protocol 3 | PASS | 1418 words |  37s | Remain

In [ ]:
# ============================================================================
# STEP 8: Create DataFrame and Export to CSV
# ============================================================================

import os

print("\nCreating results dataframe...\n")

df = pd.DataFrame(all_protocols)

# Display summary statistics
print("PROTOCOL GENERATION SUMMARY:")
print(f"\nTotal protocols: {len(df)}")
print(f"\nBy Disease:")
print(df.groupby("Disease").size())
print(f"\nWord Count Statistics:")
print(df["Word_Count"].describe())

print("\n" + "="*70)
print("Saving to Results folder...")

# Save to Results folder (relative path - much simpler now)
csv_filename = "Results/rare_disease_protocols.csv"
df.to_csv(csv_filename, index=False)

print(f"Saved: {csv_filename}")
print(f"   Rows: {len(df)}")
print(f"   Columns: {len(df.columns)}")
print(f"   Full path: {os.path.abspath(csv_filename)}")


Creating results dataframe...

PROTOCOL GENERATION SUMMARY:

Total protocols: 15

By Disease:
Disease
Autism with Microbiome Dysbiosis    3
Autoimmune Encephalitis             3
Dravet Syndrome                     3
Fabry Disease                       3
Pediatric DIPG                      3
dtype: int64

Word Count Statistics:
count      15.000000
mean     1375.066667
std       264.856691
min       586.000000
25%      1390.000000
50%      1470.000000
75%      1511.000000
max      1607.000000
Name: Word_Count, dtype: float64

Saving to Results folder...
Saved: Results/rare_disease_protocols.csv
   Rows: 15
   Columns: 9
   Full path: /content/drive/MyDrive/Colab Notebooks/research_codebase/LLM-Generated Protocols for Rare Diseases/Phase 2 - LLM & Prompt Engineering/Results/rare_disease_protocols.csv


In [ ]:
# ============================================================================
# STEP 9: Display Sample Protocol

print("\n" + "="*70)
print("SAMPLE PROTOCOL (First Generated Protocol)")
print("="*70 + "\n")

sample = df.iloc[0]
print(f"Disease: {sample['Disease']}")
print(f"Protocol ID: {sample['Protocol_ID']}")
print(f"Word Count: {sample['Word_Count']}")
print(f"\n" + "-"*70)
print(sample['Protocol_Text'][:1000] + "\n...[TRUNCATED FOR DISPLAY]...")
print("-"*70)


SAMPLE PROTOCOL (First Generated Protocol)

Disease: Fabry Disease
Protocol ID: Fabry_Disease_1
Word Count: 1024

----------------------------------------------------------------------
Clinical Trial Protocol: Fabry Disease

Introduction:
Fabry Disease (FD) is an X-linked lysosomal storage disorder (LSD) caused by GLA gene mutation. Affected individuals experience cardiovascular, renal, and neurological manifestations. Currently, enzyme replacement therapy (ERT) and substrate reduction therapy are the standard treatments. Despite these therapies, many patients still experience significant morbidity and mortality due to disease progression.

This clinical trial aims to evaluate a novel therapeutic approach for FD. The primary objective is to assess the efficacy and safety of the intervention in reducing cardiovascular and renal disease burden. Secondary objectives include assessing improvements in neurological function and quality of life.

Understanding Fabry Disease:
FD is characteri

In [ ]:
# ============================================================================
# STEP 10: Download Instructions

print("\n" + "="*70)
print("HOW TO DOWNLOAD YOUR RESULTS")
print("="*70)
print("""
Your protocols have been saved to: rare_disease_protocols.csv

To download:
1. Left sidebar → Files icon (folder icon)
2. Find "rare_disease_protocols.csv"
3. Right-click → Download
4. Save to your computer

The CSV contains:
- Disease name
- Protocol number (1, 2, or 3)
- LLM used (Mistral-7B)
- Full protocol text
- Word count
- Generation timestamp

======================================================================
ALL DONE!
======================================================================
""")


HOW TO DOWNLOAD YOUR RESULTS

Your protocols have been saved to: rare_disease_protocols.csv

To download:
1. Left sidebar → Files icon (folder icon)
2. Find "rare_disease_protocols.csv"
3. Right-click → Download
4. Save to your computer

The CSV contains:
- Disease name
- Protocol number (1, 2, or 3)
- LLM used (Mistral-7B)
- Full protocol text
- Word count
- Generation timestamp

ALL DONE!

